# Exercise 16 - Character-Level RNN

Estimated time: **30 minutes**

The goal of this exercise is to become familiar with the code for a simple character-level RNN. You can experiment with using a single-layer LSTM or using a GRU instead of an LSTM. You can also experiment with varying the choice or length of input sequences, or replacing the alphabetic sequence with a passage of English text.

This example was based on code from [Machine Learning Mastery](https://machinelearningmastery.com/understanding-stateful-lstm-recurrent-neural-networks-python-keras/).   

Recommended Hardware accelerator: **T4 GPU**

First generate some sentences (sequences of characters) that have a deterministic structure such that it is always possible to predict the next character in the sequence. The fully-trained RNN should approach 100% accuracy:

In [ ]:
import random
import numpy as np

def generate_sentence():
    sentence = []

    r = random.randint(1,5)
    max = 5
    num = 0
    count = 0
    for i in range(random.randint(1,35)):
        sentence.append('-')
        if count > 0:
            count -= 1
            if count == 0:
                sentence[-1] = str(num)
            else:
                sentence[-1] = '-'
        else:
            r += 1
            sentence[-1] = str(r)
            num = r
            count = r + 1

    if len(sentence) < 2:
        sentence = generate_sentence()

    return ''.join(sentence)

# The number of input sequences
n_seq = 1000

sentences = []
for i in range(n_seq):
    sentences.append(generate_sentence())

min_seq_length = min([len(sentences[i]) for i in range(len(sentences))])
max_seq_length = max([len(sentences[i]) for i in range(len(sentences))])
print('min_seq_length =', min_seq_length)
print('max_seq_length =', max_seq_length)

# Print a few example sequences
for i in range(20):
    print(sentences[i])

In [ ]:
import random
import numpy as np
import tensorflow
from tensorflow.keras.utils import to_categorical

tensorflow.keras.backend.clear_session()

# Define a random seed so that the run can be reproduced
np.random.seed(7)

# The alphabet is the set of possible characters used in the sentences
alphabet = "-123456789"

# Create bidirectional mappings between the characters of the alphabet and the integers
char_to_int = dict((c, i) for i, c in enumerate(alphabet))
int_to_char = dict((i, c) for i, c in enumerate(alphabet))

# Generate the set of training sequences
# Each sample consists of a variable-length input sequence, where shorter sequences are padded out,
# and a single output character, which is the next character in the sequence

padding_ch = len(alphabet)
x_data = []
y_data = []
for i in range(n_seq):
    seq = sentences.pop()
    seq_in = seq[0:-1]
    seq_out = seq[-1]
    n = len(seq_in)
    x_data.append([padding_ch] * (max_seq_length-n) + [char_to_int[char] for char in seq_in])
    y_data.append(char_to_int[seq_out])

# Convert all the input and the output characters to one-hot encoding
# Add an extra input category to serve as a padding character for short sequences
X = to_categorical(x_data, len(alphabet)+1)
y = to_categorical(y_data, len(alphabet))

X = np.reshape(X, (len(x_data), max_seq_length, len(alphabet)+1))

# Zero-out all the padding characters so that they don't contribute to any weighted sum
X[:,:,padding_ch] = 0

print('X.shape', X.shape)
print('y.shape', y.shape)

Build, train, and evaluate the Keras LSTM model. Start with the default settings for the hyperparameters, then experiment by changing the hyperparameter values.

In [ ]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Input

def build_and_train_model(n_hidden = 32, n_epochs = 1, batch_size = 1):
    global model
    model = Sequential()
    model.add(Input(shape=(X.shape[1], X.shape[2])))
    model.add(LSTM(n_hidden))
    model.add(Dense(y.shape[1], activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.summary()

    model.fit(X, y, epochs=n_epochs, batch_size=batch_size)

    loss_and_acc = model.evaluate(X, y, verbose=0)
    print(f"Accuracy = {loss_and_acc[1]*100:5.2f}")

In [ ]:
build_and_train_model()

In [ ]:
# Show some predictions of the trained model
count = 0
for i in range(len(x_data)):
    pattern = x_data[i]
    y_truth = y_data[i]
    x = np.array(pattern)
    x = to_categorical(x, len(alphabet)+1)
    x = np.reshape(x, (1, len(pattern), len(alphabet)+1))

    prediction = model.predict(x, verbose=0)

    # Choose the character with the highest probability from the softmax vector
    index = np.argmax(prediction)
    result = int_to_char[index]

    # Convert padding to spaces
    seq_in = ''.join([' ' if value == padding_ch else int_to_char[value] for value in pattern])
    print(seq_in, ">", result, '(expected', int_to_char[y_truth], ')')
    count += 1
    if count > 20: break

Now experiment with the number of input sequences, the number of hidden units, the number of epochs and the batch size. See if you can train the network to approach 100% accuracy. Investigate the interaction between the number of epochs and the batch size to see which combination works best (with respect to accuracy and/or training time) for this problem.

In [ ]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Input

def build_and_train_gru_model(n_hidden = 32, n_epochs = 1, batch_size = 1):
    global gru_model
    gru_model = Sequential()
    gru_model.add(Input(shape=(X.shape[1], X.shape[2])))
    gru_model.add(GRU(n_hidden, return_sequences=True))
    gru_model.add(GRU(n_hidden))
    gru_model.add(Dense(y.shape[1], activation='softmax'))
    gru_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    gru_model.summary()

    gru_model.fit(X, y, epochs=n_epochs, batch_size=batch_size)

    loss_and_acc = gru_model.evaluate(X, y, verbose=0)
    print(f"Accuracy = {loss_and_acc[1]*100:5.2f}")

In [ ]:
build_and_train_gru_model(32,10,5)

In [ ]:
# Show some predictions of the trained GRU model
count = 0
for i in range(len(x_data)):
    pattern = x_data[i]
    y_truth = y_data[i]
    x = np.array(pattern)
    x = to_categorical(x, len(alphabet)+1)
    x = np.reshape(x, (1, len(pattern), len(alphabet)+1))

    prediction = gru_model.predict(x, verbose=0)

    # Choose the character with the highest probability from the softmax vector
    index = np.argmax(prediction)
    result = int_to_char[index]

    # Convert padding to spaces
    seq_in = ''.join([' ' if value == padding_ch else int_to_char[value] for value in pattern])
    print(seq_in, ">", result, '(expected', int_to_char[y_truth], ')')
    count += 1
    if count > 20: break